# **Libraries**

In [47]:
import pandas as pd
import numpy as np
from datetime import datetime

# **Load & Explore Dataset**

In [48]:
file_path = '/content/influencer_marketing_roi_dataset.csv'
df = pd.read_csv(file_path)

In [49]:
print("--- First 5 Rows of the Dataset ---")
display(df.head())

--- First 5 Rows of the Dataset ---


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
0,CAMP100000,TikTok,Fitness,Giveaway,1/1/2022 0:00,79900,1892,2834,14,1/15/2022 0:00
1,CAMP100001,YouTube,Food,Product Launch,1/2/2022 0:00,47985,437228,165,13,1/15/2022 0:00
2,CAMP100002,TikTok,Travel,Brand Awareness,1/3/2022 0:00,13875,982513,2539,5,1/8/2022 0:00
3,CAMP100003,YouTube,Food,Brand Awareness,1/4/2022 0:00,41200,213400,100,20,1/24/2022 0:00
4,CAMP100004,Instagram,Food,Giveaway,1/5/2022 0:00,96998,42501,550,28,2/2/2022 0:00


In [50]:
print("\n--- Dataset Information ---")
df.info()


--- Dataset Information ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   campaign_id             150000 non-null  object
 1   platform                150000 non-null  object
 2   influencer_category     150000 non-null  object
 3   campaign_type           150000 non-null  object
 4   start_date              150000 non-null  object
 5   engagements             150000 non-null  int64 
 6   estimated_reach         150000 non-null  int64 
 7   product_sales           150000 non-null  int64 
 8   campaign_duration_days  150000 non-null  int64 
 9   end_date                150000 non-null  object
dtypes: int64(4), object(6)
memory usage: 11.4+ MB


In [51]:
print("\n--- Statistical Summary ---")
display(df.describe())


--- Statistical Summary ---


,engagements,estimated_reach,product_sales,campaign_duration_days
count,150000.000000,150000.000000,150000.000000,150000.000000
mean,50065.296107,500239.634920,2497.730433,14.976127
std,28847.302798,288003.388219,1443.222587,8.358356
min,100.000000,1002.000000,0.000000,1.000000
25%,25060.500000,251439.500000,1248.000000,8.000000
50%,50100.500000,500073.500000,2501.000000,15.000000
75%,75045.250000,750016.750000,3746.000000,22.000000
max,99999.000000,999992.000000,4999.000000,29.000000


# **Milestone 2**

In [52]:
df = df.copy()
np.random.seed(42)

print("Shape: ", df.shape)
print("\nMissed Data:\n", df.isnull().sum())
print("Duplicated Data: ", df.duplicated().sum())
df.head()

Shape:  (150000, 10)

Missed Data:
 campaign_id               0
platform                  0
influencer_category       0
campaign_type             0
start_date                0
engagements               0
estimated_reach           0
product_sales             0
campaign_duration_days    0
end_date                  0
dtype: int64
Duplicated Data:  0


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
0,CAMP100000,TikTok,Fitness,Giveaway,1/1/2022 0:00,79900,1892,2834,14,1/15/2022 0:00
1,CAMP100001,YouTube,Food,Product Launch,1/2/2022 0:00,47985,437228,165,13,1/15/2022 0:00
2,CAMP100002,TikTok,Travel,Brand Awareness,1/3/2022 0:00,13875,982513,2539,5,1/8/2022 0:00
3,CAMP100003,YouTube,Food,Brand Awareness,1/4/2022 0:00,41200,213400,100,20,1/24/2022 0:00
4,CAMP100004,Instagram,Food,Giveaway,1/5/2022 0:00,96998,42501,550,28,2/2/2022 0:00


# **Ruin the Dataset**

In [53]:
idx_outliers = df.sample(20).index
df.loc[idx_outliers, "product_sales"] = df["product_sales"].max() * 10

In [54]:
# Impossible values: 10 campaigns with a negative duration
idx_neg = df.sample(10).index
df.loc[idx_neg, "campaign_duration_days"] = -5

In [55]:
# Inconsistent text: 1% of platform names in lowercase (tiktok vs TikTok)
idx_case = df.sample(frac=0.01).index
df.loc[idx_case, "platform"] = df.loc[idx_case, "platform"].str.lower()

In [56]:
# Missing values: empty cells in 3 columns
missing_plan = {"estimated_reach": 0.05, "engagements": 0.03, "platform": 0.02}
for col, frac in missing_plan.items():
    idx = df.sample(frac=frac).index
    df.loc[idx, col] = np.nan

In [57]:
# Duplicates (done LAST so the copies stay exact): 1,000 copied rows
dup_rows = df.sample(1000)
df = pd.concat([df, dup_rows], ignore_index=True)

In [58]:
df.to_csv("influencer_dirty.csv", index=False)
print("Dirty data shape:", df.shape)

Dirty data shape: (151000, 10)


## Detect the damage we created

In [59]:
# 1) Outliers
print("1) Outliers:", (df["product_sales"] > 4999).sum(), "rows")
display(df[df["product_sales"] > 4999].head())

# 2) Negative durations
print("\n2) Negative durations:", (df["campaign_duration_days"] < 0).sum(), "rows")
display(df[df["campaign_duration_days"] < 0].head())

# 3) Lowercase platform names
print("\n3) Lowercase platforms:", df["platform"].str.islower().sum(), "rows")
display(df[df["platform"].str.islower() == True].head())

# 4) Missing values
print("\n4) Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])
display(df[df.isna().any(axis=1)].head())

# 5) Duplicates: each original row appears next to its copy
print("\n5) Duplicated rows:", df.duplicated().sum())
display(df[df.duplicated(subset="campaign_id", keep=False)].sort_values("campaign_id").head(6))

1) Outliers: 20 rows


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
1876,CAMP101876,Instagram,Beauty,Brand Awareness,2/20/2027 0:00,NaN,305987.0,49990,14,3/6/2027 0:00
20216,CAMP120216,Twitter,Fitness,Brand Awareness,5/8/2077 0:00,21722.0,891687.0,49990,2,5/10/2077 0:00
21362,CAMP121362,TikTok,Food,Seasonal Sale,6/27/2080 0:00,4124.0,978277.0,49990,8,7/5/2080 0:00
41980,CAMP141980,Instagram,Fashion,Event Promotion,12/9/2136 0:00,1752.0,264318.0,49990,25,1/3/2137 0:00
46383,CAMP146383,Twitter,Fashion,Giveaway,12/29/2148 0:00,17770.0,67325.0,49990,2,12/31/2148 0:00



2) Negative durations: 10 rows


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
7813,CAMP107813,Instagram,Gaming,Event Promotion,5/24/2043 0:00,33548.0,488541.0,3977,-5,5/25/2043 0:00
50755,CAMP150755,Instagram,Food,Event Promotion,12/18/2160 0:00,84653.0,583866.0,2542,-5,1/3/2161 0:00
52761,CAMP152761,Instagram,Fitness,Event Promotion,6/16/2166 0:00,90770.0,135283.0,2435,-5,7/10/2166 0:00
69169,CAMP169169,YouTube,Fashion,Product Launch,5/20/2211 0:00,93952.0,881808.0,3805,-5,6/5/2211 0:00
72917,CAMP172917,Instagram,Beauty,Event Promotion,8/23/2221 0:00,43641.0,393587.0,789,-5,8/29/2221 0:00



3) Lowercase platforms: 1479 rows


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
12,CAMP100012,youtube,Travel,Product Launch,1/13/2022 0:00,59591.0,343970.0,1906,19,2/1/2022 0:00
68,CAMP100068,instagram,Beauty,Giveaway,3/10/2022 0:00,14954.0,694205.0,1416,8,3/18/2022 0:00
81,CAMP100081,youtube,Food,Event Promotion,3/23/2022 0:00,35056.0,246712.0,3141,14,4/6/2022 0:00
193,CAMP100193,tiktok,Beauty,Seasonal Sale,7/13/2022 0:00,88981.0,NaN,96,8,7/21/2022 0:00
206,CAMP100206,instagram,Gaming,Product Launch,7/26/2022 0:00,41618.0,543745.0,1439,20,8/15/2022 0:00



4) Missing values per column:
platform           3023
engagements        4537
estimated_reach    7538
dtype: int64


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
6,CAMP100006,TikTok,Tech,Seasonal Sale,1/7/2022 0:00,8878.0,NaN,1027,17,1/24/2022 0:00
7,CAMP100007,TikTok,Gaming,Brand Awareness,1/8/2022 0:00,NaN,644094.0,121,13,1/21/2022 0:00
13,CAMP100013,TikTok,Fashion,Product Launch,1/14/2022 0:00,NaN,696891.0,2935,9,1/23/2022 0:00
18,CAMP100018,TikTok,Food,Brand Awareness,1/19/2022 0:00,5542.0,NaN,1083,18,2/6/2022 0:00
22,CAMP100022,Twitter,Gaming,Brand Awareness,1/23/2022 0:00,NaN,596436.0,2704,6,1/29/2022 0:00



5) Duplicated rows: 1000


,campaign_id,platform,influencer_category,campaign_type,start_date,engagements,estimated_reach,product_sales,campaign_duration_days,end_date
29,CAMP100029,YouTube,Fitness,Seasonal Sale,1/30/2022 0:00,78786.0,784142.0,178,10,2/9/2022 0:00
150093,CAMP100029,YouTube,Fitness,Seasonal Sale,1/30/2022 0:00,78786.0,784142.0,178,10,2/9/2022 0:00
331,CAMP100331,Instagram,Beauty,Product Launch,11/28/2022 0:00,65277.0,761325.0,2579,4,12/2/2022 0:00
150753,CAMP100331,Instagram,Beauty,Product Launch,11/28/2022 0:00,65277.0,761325.0,2579,4,12/2/2022 0:00
694,CAMP100694,Twitter,Fitness,Giveaway,11/26/2023 0:00,36702.0,155368.0,398,14,12/10/2023 0:00
150167,CAMP100694,Twitter,Fitness,Giveaway,11/26/2023 0:00,36702.0,155368.0,398,14,12/10/2023 0:00


## Data quality analysis (we only DETECT here, nothing is changed)

In [60]:
# Missing values: how many and what percentage per column
missing = pd.DataFrame({
    "number": df.isna().sum(),
    "percentage %": (df.isna().mean() * 100).round(2)
})
print(missing[missing["number"] > 0])

# Duplicates: full-row match vs. campaign_id match.
# campaign_id is more reliable: it catches the same campaign even if one value differs.
print("\nduplicates (all columns):", df.duplicated().sum())
print("duplicates (campaign_id):", df.duplicated(subset="campaign_id").sum())

# Impossible values
print("\nnegative duration:", (df["campaign_duration_days"] <= 0).sum())

# Outliers using the IQR rule: anything above Q3 + 1.5*IQR is suspicious
q1, q3 = df["product_sales"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print("upper IQR bound for sales:", upper)
print("sales above the bound:", (df["product_sales"] > upper).sum())
print(df["product_sales"].describe())

# Text consistency: the same platform should not appear with two spellings
print()
print(df["platform"].value_counts(dropna=False))

# Save a "before" snapshot to compare with later
before_stats = {
    "rows": len(df),
    "missing_total": int(df.isna().sum().sum()),
    "duplicates": int(df.duplicated().sum()),
    "sales_max": df["product_sales"].max(),
    "sales_mean": round(df["product_sales"].mean(), 2),
    "reach_mean": round(df["estimated_reach"].mean(), 2),
    "min_duration": df["campaign_duration_days"].min(),
}
print(before_stats)

                 number  percentage %
platform           3023          2.00
engagements        4537          3.00
estimated_reach    7538          4.99

duplicates (all columns): 1000
duplicates (campaign_id): 1000

negative duration: 10
upper IQR bound for sales: 7494.0
sales above the bound: 20
count    151000.000000
mean       2504.166934
std        1543.000606
min           0.000000
25%        1249.000000
50%        2501.500000
75%        3747.000000
max       49990.000000
Name: product_sales, dtype: float64

platform
Instagram    58415
YouTube      44079
TikTok       29294
Twitter      14710
NaN           3023
instagram      597
youtube        436
tiktok         302
twitter        144
Name: count, dtype: int64
{'rows': 151000, 'missing_total': 15098, 'duplicates': 1000, 'sales_max': 49990, 'sales_mean': np.float64(2504.17), 'reach_mean': np.float64(500195.91), 'min_duration': -5}


# **Cleaning decisions**

In [61]:
df_clean = df.copy()

# 1) DUPLICATES: the same campaign_id twice means the same campaign was logged twice.
#    Keep the first copy; keeping both would give this campaign double weight in the model.
df_clean = df_clean.drop_duplicates(subset="campaign_id", keep="first")

# 2) INCONSISTENT TEXT: "tiktok" and "TikTok" are the same platform.
#    If we leave them, the model treats them as two different platforms.
df_clean["platform"] = df_clean["platform"].str.strip().str.title()
df_clean["platform"] = df_clean["platform"].replace({"Tiktok": "TikTok", "Youtube": "YouTube"})

# 3) NEGATIVE DURATION: impossible value. Instead of deleting rows, we recompute it
#    from the dates (duration = end_date - start_date, true for all clean rows),
#    so we lose no data. We use Python's datetime because pandas cannot handle
#    the very late years that exist in this dataset (see note 7).
fmt = "%m/%d/%Y %H:%M"
bad = df_clean["campaign_duration_days"] <= 0
df_clean.loc[bad, "campaign_duration_days"] = df_clean.loc[bad].apply(
    lambda r: (datetime.strptime(r["end_date"], fmt) - datetime.strptime(r["start_date"], fmt)).days,
    axis=1)

# 4) MISSING NUMBERS (estimated_reach, engagements): fill with the mean.
#    Their distribution is symmetric (mean is about the median), and dropping the rows
#    would throw away about 7% of the data along with their valid values.
for col in ["estimated_reach", "engagements"]:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

# 5) MISSING TEXT (platform): a category has no mean, and guessing a platform
#    would invent data. We label it "Unknown" instead.
df_clean["platform"] = df_clean["platform"].fillna("Unknown")

# 6) OUTLIERS (product_sales): 20 values are about 10x the real maximum (4,999),
#    while the rest of the data is spread evenly between 0 and 4,999.
#    These are not "exceptional real sales", they are impossible, so we remove them.
#    (We removed them only because we created them ourselves; in a real project
#    we would first check the source before deleting.)
df_clean = df_clean[df_clean["product_sales"] <= 4999]

# 7) IGNORED ON PURPOSE: dates go up to the year 2432. This is an error in the
#    ORIGINAL data. We do not fix it because we have no source for the correct
#    dates and the model does not use dates as a feature. We only document it.

df_clean.to_csv("influencer_clean.csv", index=False)
print("Clean shape:", df_clean.shape)

Clean shape: (149980, 10)


# **Before vs. after comparison**

In [66]:
after_stats = {
    "rows": len(df_clean),
    "missing_total": int(df_clean.isna().sum().sum()),
    "duplicates": int(df_clean.duplicated(subset="campaign_id").sum()),
    "sales_max": df_clean["product_sales"].max(),
    "sales_mean": round(df_clean["product_sales"].mean(), 2),
    "reach_mean": round(df_clean["estimated_reach"].mean(), 2),
    "min_duration": df_clean["campaign_duration_days"].min(),
}

comparison = pd.DataFrame({"Before": before_stats, "After": after_stats})
comparison["Change"] = comparison["After"] - comparison["Before"]
print(comparison)

print()
print(df_clean["platform"].value_counts())

# Sanity check: only the 20 outlier campaigns should be missing (expected: 20)
print("\ncampaign_ids missing from clean data:",
      len(set(df["campaign_id"]) - set(df_clean["campaign_id"])))

                  Before      After    Change
rows           151000.00  149980.00  -1020.00
missing_total   15098.00       0.00 -15098.00
duplicates       1000.00       0.00  -1000.00
sales_max       49990.00    4999.00 -44991.00
sales_mean       2504.17    2497.74     -6.43
reach_mean     500195.91  500213.64     17.73
min_duration       -5.00       1.00      6.00

platform
Instagram    58618
YouTube      44205
TikTok       29404
Twitter      14753
Unknown       3000
Name: count, dtype: int64

campaign_ids missing from clean data: 20
